# 第1回　ガイダンス：統計学Ⅰの復習と本講義の位置づけ
## ―― Ⅰ＝直感の敗北、Ⅱ＝集団の敗北

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

**今日もプログラミングはしない。** 各セルの左の ▶ ボタンを上から順に押して、結果を **自分の目で見る** だけでよい。

- **統計学Ⅰを受けた人**：Ⅰでやったこと（代表値・ばらつき・相関）を高速で思い出す回。
- **Ⅱから来た人**：Colab に慣れるための回。▶ を押すだけで大丈夫。

そして最後に、**Ⅱのクライマックス（第13回）の予告編** を一瞬だけ見る ―― 「賢い人を大勢集めれば正しく決められる。ただし *独立に* 判断すれば」。

---
## 0. 準備 ―― 北辰大学のデータを用意する

統計学Ⅰで使った架空のキャンパス **北辰大学** の学生データを、このノートの中で再現する（外部ファイルは要らない）。▶ を押すだけ。

In [ ]:
# グラフの日本語表示＋データ生成。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401

rng = np.random.default_rng(2026)
N = 300
学部 = rng.choice(["経済学部", "文学部", "社会福祉学部"], N, p=[0.4, 0.35, 0.25])

# 勉強時間（時間/日）と テスト点 は正の相関を持たせる（Ⅰの『相関』復習用）
勉強時間 = np.clip(rng.normal(1.5, 1.0, N), 0, None)
テスト点 = np.clip(50 + 8 * 勉強時間 + rng.normal(0, 10, N), 0, 100).round()

# 世帯年収は右に大きく歪ませる（少数の富裕層で平均が吊り上がる＝Ⅰの『代表値』復習用）
世帯年収 = (np.exp(rng.normal(6.0, 0.7, N)) + 100).round().astype(int)  # 万円

df = pd.DataFrame({
    "学部": 学部,
    "勉強時間h": 勉強時間.round(1),
    "テスト点": テスト点.astype(int),
    "世帯年収万円": 世帯年収,
})
print(f"北辰大学の学生 {len(df)} 人のデータを用意した。")
df.head()

---
## 1. Ⅰの復習①　代表値の嘘 ―― 「平均年収」に騙されるな

北辰大学の **世帯年収** を見てみよう。

**問い：この大学の『代表的な学生』の世帯年収を語るとき、平均と中央値、どちらが実態に近い？**

予想を決めてから ▶。

In [ ]:
平均 = df["世帯年収万円"].mean()
中央値 = df["世帯年収万円"].median()
print(f"世帯年収の 平均　 ： {平均:.0f} 万円")
print(f"世帯年収の 中央値： {中央値:.0f} 万円")

plt.figure(figsize=(7, 4))
plt.hist(df["世帯年収万円"], bins=40, color="#4dabb6", edgecolor="white")
plt.axvline(平均, color="#e8503a", ls="-", lw=2, label=f"平均 {平均:.0f}")
plt.axvline(中央値, color="#333", ls="--", lw=2, label=f"中央値 {中央値:.0f}")
plt.xlabel("世帯年収（万円）")
plt.ylabel("人数")
plt.title("北辰大学・世帯年収の分布")
plt.legend()
plt.show()

**平均は中央値より高く出る。** 少数の富裕世帯が平均を右へ引っ張るからだ。

分布が歪んでいるとき、「平均年収◯◯万円」は *ほとんどの学生より高い* 値になりうる。**どの代表値を選ぶかは、すでに一つの主張**だ ―― これがⅠの第2回だった。

---
## 2. Ⅰの復習②　相関 ―― ただし「相関」は「因果」ではない

**勉強時間** と **テスト点** の関係を散布図で見る。

予想：相関はあると思う？　あるとして、それは「勉強すれば点が上がる」という *因果* だと言い切れる？

In [ ]:
r = df["勉強時間h"].corr(df["テスト点"])
print(f"勉強時間とテスト点の相関係数 r = {r:.2f}")

plt.figure(figsize=(7, 4))
plt.scatter(df["勉強時間h"], df["テスト点"], s=18, alpha=0.6, color="#4dabb6")
plt.xlabel("勉強時間（時間/日）")
plt.ylabel("テスト点")
plt.title(f"勉強時間 と テスト点（r = {r:.2f}）")
plt.show()

正の相関がある。だが **相関は因果ではない**。「もともと真面目な性格」のような第三の要因（交絡）が、勉強時間とテスト点の *両方* を押し上げているだけかもしれない。

> Ⅰの第5回でやったこの問いを、Ⅱでは **第10〜11回（因果推論・交絡）** で数理的に深掘りする。観察データから因果を主張することの難しさが、今期の大きなテーマの一つだ。

---
## 3. Ⅱの予告編　―― 「賢い集団」は本当に賢いのか？

ここからが統計学Ⅱの世界だ。

次のような状況を考える：

- ある問題に「正解」がある（YES か NO か）。
- 一人ひとりは完璧ではないが、**コインより少しだけ賢い**（正解率 55%）。
- その人たちを集めて **多数決** で答えを決める。

**問い：人数を増やすと、多数決の正解率はどうなる？**　55%のまま？　それとも上がる？

予想を決めてから ▶。

In [ ]:
# 正解率55%の人を n 人集めて多数決したときの『多数決の正解率』を測る
# 各人は『独立に』自分で判断する、という前提でシミュレーションする
r2 = np.random.default_rng(42)
p個人 = 0.55          # 一人の正解率
試行 = 20000           # 何回投票をやり直すか
人数リスト = [1, 3, 5, 11, 21, 51, 101, 201]

多数決正解率 = []
for n in 人数リスト:
    投票 = r2.random((試行, n)) < p個人      # True=その人は正解した（独立に判断）
    多数派が正解 = 投票.sum(axis=1) > n / 2   # 過半数が正解なら多数決も正解
    多数決正解率.append(多数派が正解.mean())

for n, acc in zip(人数リスト, 多数決正解率):
    print(f"{n:>4} 人で多数決 → 正解率 {acc:.1%}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(人数リスト, 多数決正解率, marker="o", color="#e8503a")
plt.axhline(0.55, ls="--", color="gray", label="一人の正解率 55%")
plt.ylim(0.4, 1.0)
plt.xlabel("投票する人数")
plt.ylabel("多数決の正解率")
plt.title("独立に判断する人を増やすと、多数決はほぼ確実に正解する")
plt.legend()
plt.show()

たった55%の人たちでも、**独立に**判断して数を増やせば、多数決の正解率は **ほぼ100%** に近づく。これが **コンドルセの陪審定理**（1785年）―― 「群衆の智慧」の数学的な正体だ。

> ⚠️ **ただし、ここに罠がある**
> 
> いま一行だけ、こっそり強い仮定を置いた ―― **「各人が *独立に* 判断する」**。
> 
> もし人々が互いに顔色をうかがい、**「空気を読んで」** 多数派に合わせ始めたら？　この定理は **崩壊する**。賢いはずの集団が、全員そろって間違える。
> 
> なぜそうなるのか、どれくらい正解率が落ちるのかを、**第13回「『空気を読む』ことの愚かさ：コンドルセの陪審定理」** でシミュレーションして確かめる。今日はその予告編だ。


---
## 今日のまとめ

| | Ⅰで学んだこと | Ⅱで深掘りすること |
|---|---|---|
| 代表値 | 平均は外れ値に弱い | 確率分布の数理（第4〜5回） |
| 相関 | 相関 ≠ 因果 | 因果推論・交絡（第10〜11回） |
| 集団 | （Ⅰでは扱わない） | **独立性と集合知（第12〜13回）** |

Ⅰは「**あなたの直感**は系統的に外れる」を示した。
Ⅱは「**集団になると、もっと外れる**（空気を読むと）」を、確率論で証明していく。

今期を貫く一本の糸は **「独立性」** だ。

> **課題（Moodle）**：Ⅰの要点クイズ（自動採点）＋「Ⅰで一番『直感が外れた』のはどこか／Ⅱに何を期待するか」のふりかえり（記述）。Ⅱから来た人は『統計に対する今の自分の構え』を書く。詳しくはMoodleの第1回課題を見ること。